# Módulo 3 · Clase 5: SVM, Pipelines y Validación Honesta — Cierre del Módulo

**Machine Learning for Petroleum Engineers Using Python**
SLB Ecuador · UDLA · 2026

Instructor: **Carlos Enrique Mosquera Trujillo**
Repo: [github.com/cmosquerat/slb-diplomado](https://github.com/cmosquerat/slb-diplomado)

---
## La idea de hoy

Última clase del módulo. Un problema nuevo — **integridad de ductos**, con fallas reales y costos en dólares reportados al regulador de EE. UU. — y cuatro herramientas que convierten a un usuario de `.fit()` en un profesional:

1. **SVM**: el último modelo del módulo (la "carretera más ancha") y su **kernel trick**.
2. **Pipelines**: el tubo que ordena todo el flujo y hace imposible equivocarse al escalar.
3. **Validación honesta**: validación cruzada, `GridSearchCV` sin trampa, la **fuga de información** medida en vivo, y `GroupKFold` — con el que pagaremos la deuda que arrastramos desde la Clase 2.
4. **SMOTE**: qué hacer cuando la clase que importa (¿la fuga se enciende?) es rara.

## El problema, acotado (regla de la casa)

| | |
|---|---|
| **Pregunta de negocio** | Dada una línea y sus condiciones: ¿la falla fue **corrosión**? Y después: ¿la fuga terminó en **ignición**? |
| **Target** | `CORROSION` (sí/no); luego `IGNICION` |
| **Features** | 6 numéricas + `RECUBRIMIENTO` (categórica → one-hot, como en la C4) |
| **Métrica de éxito** | accuracy y recall — con **validación honesta** |
| **Récord a batir** | el modelo tonto (nunca corrosión): **62 %** |

> **Dataset**: incidentes reales en gasoductos de transmisión reportados a PHMSA (regulador de ductos de EE. UU.), 2010–hoy. Curados a 638 incidentes de 132 operadores con reporte completo de tubería.

---
# 0 · Preparación

Las librerías de siempre, más dos piezas nuevas: `SVC` (la SVM) y las herramientas de validación. Cada import está comentado con su papel de hoy.

In [ ]:
import pandas as pd                 # tablas
import numpy as np                  # numeros
import matplotlib.pyplot as plt    # graficas

from sklearn.model_selection import (
    train_test_split,     # separar train/test (desde C1)
    cross_val_score,      # validacion cruzada (nuevo)
    GroupKFold,           # pliegues por grupo (nuevo)
    GridSearchCV)         # busqueda de perillas (C4)
from sklearn.preprocessing import StandardScaler   # escalar (C2)
from sklearn.pipeline import make_pipeline         # el tubo (nuevo)
from sklearn.svm import SVC                        # la SVM (nuevo)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

URL = "https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/phmsa_gasoductos.csv"
d = pd.read_csv(URL)     # cargar el dataset curado
print(d.shape)

---
# 1 · Exploración: conocer los incidentes

Regla del módulo: nadie modela datos que no ha mirado.

In [ ]:
d.head()

## Las columnas, en una línea cada una

| Columna | Qué es |
|---|---|
| `DIAMETRO_PULG` | diámetro de la línea (pulgadas) |
| `EDAD_ANIOS` | años desde la instalación hasta el incidente |
| `PRESION_MAX_PSI` | presión máxima de operación permitida (MOP) |
| `PRESION_INCIDENTE_PSI` | presión al momento del incidente |
| `RECUBRIMIENTO` | alquitrán, asfalto, cinta fría, sin recubrimiento… |
| `CLASE_UBICACION` | densidad poblacional alrededor (1 rural → 4 urbana) |
| `COSTA_AFUERA` | ¿línea costa afuera? (0/1) |
| `CAUSA` / `IGNICION` | **targets**: causa investigada; ¿hubo fuego? |
| `COSTO_USD` | costo total reportado del incidente |
| `GAS_LIBERADO_MPCS`, `IND_CORROSION_GALVANICA` | ⚠️ **columnas trampa** — ya hablaremos de ellas |

In [ ]:
print(d['CAUSA'].value_counts())        # ¿por que fallan los ductos?
print()
print('Tasa de ignicion:', d['IGNICION'].mean().round(3))   # el evento raro

**Lectura**: la **corrosión** es la causa #1 (242 de 638, un 38 %) — nuestro primer target. Y la **ignición** es rara: 12 %. Guarden ese número.

In [ ]:
d[['DIAMETRO_PULG', 'EDAD_ANIOS', 'PRESION_MAX_PSI',
   'PRESION_INCIDENTE_PSI', 'COSTO_USD']].describe().round(1)

**Lectura física**: líneas de hasta 42 pulgadas y hasta 112 años de edad; presiones de operación de hasta 3 255 psi. Y miren `COSTO_USD`: el incidente mediano cuesta **$300 mil**; el máximo, cientos de millones. Estos números son reales.

In [ ]:
costo_fuego = d[d['IGNICION'] == 1]['COSTO_USD'].median()   # mediana con fuego
costo_sin = d[d['IGNICION'] == 0]['COSTO_USD'].median()      # mediana sin fuego
print(f'Incidente mediano CON ignicion: ${costo_fuego:,.0f}')
print(f'Incidente mediano sin ignicion: ${costo_sin:,.0f}')

Casi **4 veces más caro** cuando hay fuego — sin contar personas. Esa es la segunda pregunta de negocio, para el final.

## ¿La física habla? Edad y recubrimiento

In [ ]:
d['CORROSION'] = (d['CAUSA'] == 'CORROSION').astype(int)   # target binario 0/1

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(d[d.CORROSION == 0]['EDAD_ANIOS'], bins=24, alpha=0.6,
             color='gray', label='otras causas')
axes[0].hist(d[d.CORROSION == 1]['EDAD_ANIOS'], bins=24, alpha=0.6,
             color='#C82B40', label='corrosion')
axes[0].set_xlabel('Edad de la linea (años)'); axes[0].legend()
axes[0].set_title('¿La corrosion es cosa de lineas viejas?')

tasa = (d.groupby('RECUBRIMIENTO')['CORROSION'].mean() * 100).sort_values()
axes[1].barh(tasa.index, tasa.values, color='#C82B40')
axes[1].set_xlabel('% de incidentes que fueron corrosion')
axes[1].set_title('El recubrimiento cuenta una historia')
plt.tight_layout(); plt.show()

**Lectura**: sorpresa — la corrosión **no** es simplemente "línea vieja" (su pico está en los 40 años). Pero el **recubrimiento** sí ordena el riesgo: cinta fría 12 % de corrosión, sin recubrimiento 48 %. La física responde: el recubrimiento es la primera barrera anticorrosiva.

> 🔧 **Mini-ejercicio 1**: calcula el **costo mediano** (`COSTO_USD`) de los incidentes por corrosión vs los demás. ¿Cuál grupo es más caro? (Pista: filtra con `d[d.CORROSION == 1]`.)

In [ ]:
# Escribe tu solucion aqui

## ⚠️ Las dos columnas trampa

`GAS_LIBERADO_MPCS` (cuánto gas escapó) y `IND_CORROSION_GALVANICA` (lo marca el investigador **cuando ya determinó** que hubo corrosión) solo se conocen **después** de la falla.

> 🤔 **Pregunta clave**: si el objetivo es priorizar inspecciones **antes** de que la línea falle… ¿podemos usar columnas que solo existen **después**? *No: sería predecir el pasado. Se llama **fuga de información**. Más adelante la mediremos en vivo.*

---
# 2 · Preparar los datos (todo lo aprendido en la C4)

Codificamos `RECUBRIMIENTO` con one-hot y separamos train/test estratificado. Nada nuevo — repaso exprés.

In [ ]:
onehot = pd.get_dummies(d['RECUBRIMIENTO'], prefix='Rec')   # one-hot (C4)

FEATURES = ['DIAMETRO_PULG', 'EDAD_ANIOS', 'PRESION_MAX_PSI',
            'PRESION_INCIDENTE_PSI', 'CLASE_UBICACION', 'COSTA_AFUERA']
X = pd.concat([d[FEATURES], onehot], axis=1)   # 6 numericas + one-hot
y = d['CORROSION'].values                      # el target

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)   # stratify (C4)
print('Train:', X_tr.shape, '| Test:', X_te.shape)

In [ ]:
pred_tonto = np.zeros_like(y_te)          # el tonto: "nunca es corrosion"
print('Modelo tonto:', round(accuracy_score(y_te, pred_tonto), 3))

**0.62 es el récord a batir.**

---
# 3 · SVM: la carretera más ancha

**La idea, sin matemática**: entre dos poblados de puntos hay muchas rectas que los separan. La SVM elige la que deja **la carretera más ancha** posible entre los dos — así, un dato nuevo ligeramente distinto sigue cayendo del lado correcto. Los puntos que "tocan la cerca" son los **vectores de soporte**: ellos solos definen la frontera.

En código es el patrón de siempre:

In [ ]:
svm = SVC()                  # el modelo SVM (kernel rbf por defecto)
svm.fit(X_tr, y_tr)          # aprende del train
print(svm.score(X_te, y_te)) # acierto en el test

**0.776** — ya le gana al tonto. Pero este modelo trabaja midiendo **distancias** entre incidentes ("¿qué tan parecidas son estas dos líneas?")… y nuestras variables tienen escalas muy distintas. Veamos cuánto nos cuesta eso.

In [ ]:
rangos = X_tr[FEATURES].max() - X_tr[FEATURES].min()   # rango de cada variable
print(rangos.round(1))

La presión se mueve en un rango de **3 205 psi**; la edad, en **112 años**; costa afuera, en **1**. Para una distancia, la presión **grita** y las demás susurran. Es "el paso parejo" de la Clase 1: hay que **estandarizar** para que todas opinen con la misma voz.

## Escalar a mano funciona… hasta que alguien se equivoca

En la Clase 2 escalamos a mano: ajustar el escalador con el train, transformar el train, transformar el test con el **mismo** escalador, y no confundirse nunca. Dos errores clásicos acechan: evaluar el test sin escalar, o ajustar el escalador con **todos** los datos (fuga).

---
# 4 · Pipelines: el tubo

Piensen en su **tren de tratamiento**: el crudo entra crudo, pasa por el separador, el deshidratador, la bomba — y sale listo. Nadie carga baldes entre equipos. Un **pipeline** es eso: las piezas encadenadas en un solo objeto con `.fit()` y `.predict()`.

In [ ]:
tubo = make_pipeline(
    StandardScaler(),    # paso 1: escalar
    SVC())               # paso 2: clasificar
tubo.fit(X_tr, y_tr)     # el tubo entero aprende junto
print(tubo.score(X_te, y_te))

De 0.776 a **0.823**: casi **5 puntos gratis** por escalar bien — y con *menos* código que la versión manual.

**Los 3 porqués del pipeline** (de aquí en adelante, todo modelo con preparación va en tubo):

1. **Código más simple** — un objeto, menos variables sueltas.
2. **Cero olvidos** — el test siempre pasa por el mismo tratamiento que el train.
3. **Validación honesta** — cuando hagamos validación cruzada, el escalador se reajustará *dentro de cada pliegue*, sin fugas. A mano casi nadie lo hace bien.

> 🔧 **Mini-ejercicio 2**: armen un `tubo_logistica` con `StandardScaler()` y `LogisticRegression(max_iter=2000)`, entrénenlo y compárenlo con el SVM. ¿Quién va ganando?

In [ ]:
# Escribe tu solucion aqui

---
# 5 · El kernel trick: subir de dimensión

¿Y si **ninguna recta** puede separar? Fabriquemos un caso extremo: un anillo que rodea a un centro (`make_circles`).

In [ ]:
from sklearn.datasets import make_circles

Xc, yc = make_circles(n_samples=300, noise=0.07,
                      factor=0.45, random_state=42)   # datos de juguete

plt.figure(figsize=(5, 5))
plt.scatter(Xc[yc == 0, 0], Xc[yc == 0, 1], c='#2563EB', s=14)
plt.scatter(Xc[yc == 1, 0], Xc[yc == 1, 1], c='#C82B40', s=14)
plt.title('¿Que recta separa esto? Ninguna')
plt.show()

svm_lineal = SVC(kernel='linear').fit(Xc, yc)   # SVM sin curvas
print('SVM lineal:', round(svm_lineal.score(Xc, yc), 2))

**58 %** — apenas una moneda al aire. Ahora el movimiento clave: inventemos **a mano** una columna nueva, la **distancia al centro** (z = x² + y²), y miremos en 3D.

In [ ]:
z = Xc[:, 0]**2 + Xc[:, 1]**2       # la columna nueva: distancia al centro

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection='3d')
ax.scatter(Xc[yc == 0, 0], Xc[yc == 0, 1], z[yc == 0], c='#2563EB', s=14)
ax.scatter(Xc[yc == 1, 0], Xc[yc == 1, 1], z[yc == 1], c='#C82B40', s=14)
ax.set_zlabel('z = x2 + y2')
ax.set_title('Con la dimension nueva, un PLANO los separa')
plt.show()

Los rojos quedaron **abajo** y los azules **arriba**: un simple plano basta. Crear la columna correcta **convierte un problema curvo en uno recto**.

**El problema**: ¿cuál columna inventar? ¿x²? ¿x·y? Con nuestras 11 features del dataset real, solo los cuadrados y productos ya son más de 70 columnas — y quizá se necesiten cientos.

**El kernel trick**: una propiedad matemática de la SVM le permite lograr **el efecto** de todas esas columnas **sin calcular ninguna**. En el código, todo el truco es `kernel='rbf'`… que es el valor por defecto.

In [ ]:
svm_rbf = SVC(kernel='rbf').fit(Xc, yc)   # el kernel hace el trabajo
print('SVM con kernel rbf:', round(svm_rbf.score(Xc, yc), 2))

**100 %**, sin que nadie fabricara la columna. Ese es el truco.

> 🔧 **Mini-ejercicio 3**: repite el experimento con las **lunas** de la Clase 3 (`from sklearn.datasets import make_moons`, con `noise=0.2, random_state=42`): compara `SVC(kernel='linear')` vs `SVC(kernel='rbf')`. ¿Cuánto gana el kernel?

In [ ]:
# Escribe tu solucion aqui

## Las perillas de la SVM: `gamma` y `C`

- **`gamma`** = el *radio de influencia* de cada punto. Pequeño → frontera suave, casi recta. Grande → cada punto solo manda en su vecindario y la frontera se vuelve paranoica (encierra puntos sueltos = **memorizar**, el overfitting de la C3).
- **`C`** = el *castigo por invadir el margen*. Chico → carretera ancha que tolera errores. Grande → la frontera se retuerce para no dejar ni un invasor.

Son los **hiperparámetros** de la SVM (las perillas de la C4). ¿Y en el problema real de corrosión, gana el kernel?

In [ ]:
tubo_lineal = make_pipeline(StandardScaler(), SVC(kernel='linear'))
tubo_lineal.fit(X_tr, y_tr)                  # SVM sin curvas, en tubo
print('SVM lineal:', round(tubo_lineal.score(X_te, y_te), 3))
print('SVM rbf   :', round(tubo.score(X_te, y_te), 3))

**La lección honesta**: el kernel *no* ganó (0.828 vs 0.823) — el problema de corrosión resultó casi lineal. Pasa muchísimo con datos tabulares reales. Por eso un profesional **siempre prueba el lineal primero**: el kernel es un as bajo la manga (círculos, lunas), no un reflejo automático.

---
# 6 · Validación honesta

## Una foto puede mentir: 5 fotos mienten menos

Nuestro train/test es **una** repartición al azar. ¿Y si nos tocó una fácil? `cross_val_score` reparte los datos en 5 pliegues y hace 5 exámenes distintos:

In [ ]:
notas = cross_val_score(tubo, X, y, cv=5)   # 5 examenes distintos
print(notas.round(3))            # las 5 notas
print(notas.mean().round(3))     # la nota promedio

Miren la dispersión: de 0.764 a 0.812. **Una sola foto pudo darnos cualquiera de esos números.** Reportar la media de 5 es más honesto — y esto es exactamente lo que `GridSearchCV` hacía por dentro en la Clase 4 (`cv=5`).

## GridSearch sobre el tubo completo

Novedad: cuando el pipeline va dentro de la búsqueda, la parrilla nombra la pieza — `svc__C` significa "la perilla `C`, de la pieza `svc`". Y el escalador se reajusta **dentro de cada pliegue**: cero fugas (porqué #3 del pipeline, cumplido).

In [ ]:
perillas = {'svc__C': [0.1, 1, 10, 100],     # castigo por invadir
            'svc__gamma': [0.01, 0.1, 1]}    # radio de influencia

buscador = GridSearchCV(tubo, perillas, cv=5)   # 12 combos x 5 pliegues
buscador.fit(X_tr, y_tr)
print('Mejor combinacion:', buscador.best_params_)
print('Test con lo mejor:', round(buscador.score(X_te, y_te), 3))

El ganador (`C=1, gamma=0.01` → test 0.828) usa el gamma **más suave** de la parrilla — otra pista de que el problema es casi lineal. Y noten: mejora fina (0.823 → 0.828), sin trampa. No siempre hay tesoro escondido, ya lo sabíamos de la C4.

## La trampa de la fuga, medida en vivo

¿Recuerdan las columnas prohibidas? Agreguemos UNA "por curiosidad":

In [ ]:
X_trampa = X.copy()
X_trampa['GALV'] = d['IND_CORROSION_GALVANICA']   # columna POST-investigacion

a_tr, a_te, b_tr, b_te = train_test_split(
    X_trampa, y, test_size=0.3, random_state=42, stratify=y)
tubo_trampa = make_pipeline(StandardScaler(), SVC())
tubo_trampa.fit(a_tr, b_tr)
print('Con la columna trampa:', round(tubo_trampa.score(a_te, b_te), 3))

¡De 0.823 a **0.906**! ¿Ocho puntos gratis? No: un modelo que necesita **que la falla ya haya ocurrido y esté investigada**. Inútil para priorizar inspecciones. **Predice el pasado.**

Así se ve la **fuga de información** en la vida real: un número espectacular que se derrumba en producción. Ya la vimos con 3 disfraces en el módulo: escalar antes del split (hoy), imputar con todo el dataset (C4), y esta. Si un resultado parece demasiado bueno, pregunten **qué sabía el modelo y cuándo**.

## GroupKFold: ¿el modelo viaja?

Nuestros pliegues mezclan operadores. Pero el negocio pregunta: ¿funciona con un operador **nuevo**?

In [ ]:
notas_grupo = cross_val_score(
    tubo, X, y,
    cv=GroupKFold(5),            # pliegues que dejan grupos completos fuera
    groups=d['OPERADOR_ID'])     # el grupo: cada operador
print('Operadores por fuera:', notas_grupo.mean().round(3))

0.787 → **0.766**: cae solo 2 puntos. Buena noticia — el modelo aprendió física de ductos, no mañas de cada operador. **Viaja.**

## La deuda del módulo: ¿y las facies?

Desde la Clase 2 venimos anotando: "validar con pozos completos por fuera". Hagámoslo — con el dataset de facies de la C4:

In [ ]:
facies = pd.read_csv("https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/hugoton_facies.csv").dropna()   # el dataset de la C4
X_fac = pd.concat([
    facies[['GR', 'ILD_log10', 'DeltaPHI', 'PHIND', 'PE', 'RELPOS']],
    (facies['NM_M'] == 2).astype(int).rename('Marino'),      # binarizar (C4)
    pd.get_dummies(facies['Formation'], prefix='Fm')], axis=1)  # one-hot (C4)
y_fac = facies['Facies'].values     # target multiclase
bosque = RandomForestClassifier(n_estimators=200, random_state=42)

from sklearn.model_selection import KFold
al_azar = cross_val_score(bosque, X_fac, y_fac,
                          cv=KFold(5, shuffle=True, random_state=42))
por_pozo = cross_val_score(bosque, X_fac, y_fac, cv=GroupKFold(5),
                           groups=facies['Well Name'])   # POZOS por fuera
print('Pliegues al azar :', al_azar.mean().round(3))
print('Pozos por fuera  :', por_pozo.mean().round(3))

**0.78 → 0.53. Veinticinco puntos de espejismo.** Medios pies vecinos del mismo pozo estaban repartidos entre train y test — el modelo "adivinaba" porque ya conocía el pozo. El 0.76 que celebramos en la C4 era real *solo para pozos ya vistos*.

`GroupKFold` no "mejora" el modelo: **les dice la verdad**. A veces certifica (ductos), a veces desinfla (facies). Ambas valen oro.

---
## 🧩 Práctica 1: afinar con honestidad (15 min)

Todo con pipeline. Todo sin tocar el test hasta el final.

1. Armen un pipeline con `StandardScaler` y `SVC(kernel='linear')` y calculen su `cross_val_score` con `cv=5`. ¿Confirma que el lineal compite con el rbf?
2. Amplíen la parrilla del `GridSearchCV` agregando `'svc__kernel': ['linear', 'rbf']`. ¿Quién gana ahora y con qué perillas?
3. Evalúen su mejor modelo con `GroupKFold` por operador. ¿Cuánto cae respecto a los pliegues aleatorios?
4. **Pregunta de negocio**: escriban una frase para el gerente de integridad diciendo qué accuracy debe esperar **y por qué esa** (¿la de pliegues aleatorios o la de operadores por fuera?).

In [ ]:
# Escribe tu solucion aqui

---
# 7 · SMOTE: cuando lo importante es raro

## La segunda pregunta de negocio: el fuego

Solo el **12 %** de las fugas se encienden — pero el incidente mediano con fuego cuesta **$896 000** contra $234 000 sin fuego. Si pudiéramos anticipar qué líneas, al fallar, tienden a encenderse, priorizaríamos válvulas de cierre automático y planes de emergencia.

> 🤔 **Pregunta clave**: el tonto ("nunca hay fuego") acierta el 88 %. ¿Recuerdan al modelo perezoso de la C4, ciego a la dolomita? ¿Qué hará nuestro SVM con los fuegos? Formulen su hipótesis antes de correr la celda.

In [ ]:
y_fuego = d['IGNICION'].values      # nuevo target: ¿hubo fuego?
X_tr2, X_te2, yf_tr, yf_te = train_test_split(
    X, y_fuego, test_size=0.3, random_state=42, stratify=y_fuego)

tubo_fuego = make_pipeline(StandardScaler(), SVC())
tubo_fuego.fit(X_tr2, yf_tr)                     # mismo tubo, target fuego
pred = tubo_fuego.predict(X_te2)
print('Accuracy:', round(accuracy_score(yf_te, pred), 3))
print('Recall del fuego:', round(recall_score(yf_te, pred), 3))

Accuracy **0.88**… y recall **0.000**: de los 23 fuegos del test, detectó **cero**. Con 12 % de fuegos, ignorarlos por completo *ya da* 88 % — la accuracy **premia la ceguera**.

El modelo casi no ve fuegos al entrenar: 53 contra 393. ¿Y si le **fabricamos** más ejemplos?

## SMOTE: fabricar casos parecidos a los raros

SMOTE toma cada fuego real, busca su fuego **más parecido**, y crea puntos **intermedios** entre los dos: casos sintéticos pero creíbles, en la misma zona del espacio.

**Regla sagrada**: SMOTE se aplica **SOLO al train** — inventar datos en el examen sería otra fuga. El pipeline de `imblearn` la cumple automáticamente (aplica SMOTE al entrenar y lo omite al predecir).

In [ ]:
from imblearn.over_sampling import SMOTE               # fabricar minoritarios
from imblearn.pipeline import make_pipeline as make_pipeline_imb

tubo_smote = make_pipeline_imb(
    StandardScaler(),          # escalar
    SMOTE(random_state=42),    # equilibrar (SOLO el train)
    SVC())                     # clasificar
tubo_smote.fit(X_tr2, yf_tr)
pred_smote = tubo_smote.predict(X_te2)
print('Accuracy:', round(accuracy_score(yf_te, pred_smote), 3))
print('Recall del fuego:', round(recall_score(yf_te, pred_smote), 3))

**El despertar**: recall 0.00 → **0.35** (detecta 8 de 23 fuegos). El precio: la accuracy baja a 0.72. Veamos el detalle:

In [ ]:
print(confusion_matrix(yf_te, pred_smote))   # filas = real [sin fuego, fuego]

**Cómo leerla**: detecta 8 fuegos (abajo-derecha), se le escapan 15, y genera **39 falsas alarmas** (arriba-derecha) — 1 de cada 5–6 alertas es fuego real.

¿Vale la pena? Con fuego a **$896 000** y falsa alarma al costo de una revisión preventiva, la cuenta sale sola. Es el trade-off del **umbral** de la Clase 2, con otra herramienta.

> 🔧 **Mini-ejercicio 4**: verifiquen la regla sagrada — impriman cuántos fuegos hay en el train antes de SMOTE (`yf_tr.sum()`) y cuántos fabricó SMOTE: usen `SMOTE(random_state=42).fit_resample(X_tr2, yf_tr)` y cuenten con `np.bincount`. ¿Quedó 393 contra 393?

In [ ]:
# Escribe tu solucion aqui

---
## 🧩 Práctica 2: domar el fuego (15 min)

1. Armen `RandomForestClassifier(n_estimators=200, random_state=42)` + SMOTE (con el pipeline de `imblearn`, **sin** escalador: los árboles no lo necesitan). Comparen accuracy y recall contra el SVM + SMOTE.
2. Prueben `SMOTE(sampling_strategy=0.5, random_state=42)` — fabricar fuegos hasta la **mitad** de la clase grande, no hasta igualar. ¿Mejora el trato recall/falsas alarmas?
3. Saquen la `confusion_matrix` de su mejor opción y cuenten: ¿cuántos fuegos detecta? ¿cuántas falsas alarmas genera?
4. **Pregunta de negocio**: con fuego = $896 000 y revisión preventiva = $20 000, escriban una frase para el gerente HSE recomendando (o no) desplegar su modelo.

In [ ]:
# Escribe tu solucion aqui

---
# Cierre del Módulo 3

## Lo que aprendimos hoy

- **SVM**: margen máximo, vectores de soporte — y el **kernel trick** para subir de dimensión sin pagar el costo.
- **Pipelines**: honestidad automatizada (código simple, cero olvidos, CV sin fugas).
- **Validación cruzada**: 5 fotos mienten menos que una.
- **Fuga de información** y sus 3 disfraces — la medimos en vivo (0.823 → 0.906 falsos).
- **GroupKFold**: ¿el modelo viaja? Ductos sí (−2 pts); facies no (**−25 pts**: el número del día).
- **SMOTE**: fabricar lo raro, solo en el train — recall del fuego 0.00 → 0.35.

## El mapa de modelos del módulo

| Modelo | Su fuerte | ¿Escalar? |
|---|---|---|
| Lineal / Logística | explicable, rápida — empiecen aquí | sí |
| Árbol / Random Forest | no linealidad sin esfuerzo | no |
| SVM | margen máximo; kernel para lo curvo | **sí, siempre** |

Receta profesional: simple primero, bosque después, SVM cuando el problema lo pida — y **siempre** contra el tonto. Para su radar: *gradient boosting* (XGBoost) y *k-NN*; con lo de hoy pueden aprenderlos solos.

## El arco de las 5 clases

**C1** predecir un número → **C2** predecir una etiqueta + costo → **C3** romper la recta → **C4** elegir entre muchas → **C5** hacerlo **honesto**.

> Ya no son personas que corren `.fit()`: son ingenieros que saben **cuándo creerle a un modelo**.